<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Director-Fix/mnps_job_classification_two_pass_gpt-4o_v755.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Classification_gpt-4o (v7.5.5 - Director Fix)**
> Two-Pass Self-Consistency with Director Classification Enhancement  
> DSI DSSG + MNPS   

## **Setup Requirements**
1. **OpenAI API Key**: Add your key to Colab Secrets (🔑 icon in left sidebar)
   - Secret name: `OPENAI_API_KEY`
   - Secret value: Your OpenAI API key
2. **Upload Files**: 
   - `MNPS Prompt Resources.zip`
   - `Sample JDs.csv`

## **Version 7.5.5 Changes**
- **All v7.5.4 features**: Two-pass classification with self-consistency check
- **NEW - Director Classification Fix**: 
  - Added explicit Director vs Manager vs Supervisor distinction
  - New `identify_director_role()` function with weighted scoring
  - Enhanced prompt guidelines for Director identification
  - Improved executive role level assignment (I, II, III)
- **Expected Impact**: Fixes Director misclassification (was classified as Manager III)
- **Target**: Progress toward 77-79% combined accuracy for production deployment

In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")

    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")

In [ ]:
# ==== 2) Load problem role cheat sheet and define closed sets ====
# Build major role list from ground truth
MNPS_MAJOR_ROLES = sorted(gt_df['major_role_group'].dropna().unique().tolist())
print(f"✅ Found {len(MNPS_MAJOR_ROLES)} unique major roles from ground truth")

# Define minor roles (closed set)
MINOR_ROLES = ['I', 'II', 'III', 'Lead']

# Define executive roles (rarely have "Lead" sub-grouping)
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager', 'Assistant Principal']

# Normalization mappings
MINOR_NORMALIZATION = {
    '1': 'I', 'i': 'I', 'one': 'I',
    '2': 'II', 'ii': 'II', 'two': 'II',
    '3': 'III', 'iii': 'III', 'three': 'III',
    'lead': 'Lead', 'LEAD': 'Lead', 'senior': 'Lead', 'principal': 'Lead'
}

def normalize_minor(minor: str) -> str:
    """Normalize minor sub-group to standard format."""
    if not minor or pd.isna(minor):
        return 'I'
    minor_str = str(minor).strip()
    # Direct match
    if minor_str in MINOR_ROLES:
        return minor_str
    # Normalized match
    normalized = MINOR_NORMALIZATION.get(minor_str.lower(), minor_str)
    # Ensure it's in our closed set
    if normalized not in MINOR_ROLES:
        # Try extracting Roman numerals
        if 'III' in normalized.upper():
            return 'III'
        elif 'II' in normalized.upper():
            return 'II'
        elif 'I' in normalized.upper():
            return 'I'
        elif 'lead' in normalized.lower() or 'senior' in normalized.lower():
            return 'Lead'
        # Default to I
        return 'I'
    return normalized

print("✅ Defined closed sets and normalization rules for major and minor roles")

In [ ]:
# ==== 3) Enhanced normalization helpers with Problem Role Cheat Sheet logic ====

# Problem roles with fallback patterns
SPECIALIST_FALLBACKS = [
    ('Technician', r'(technician|technical|equipment|repair|maintenance|troubleshoot)'),
    ('Analyst', r'(analyst|analysis|data|research|evaluation|assessment|statistical)'),
    ('Architect', r'(architect|architectural|design|construction|building|space planning)')
]

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Discourage 'Specialist' by checking for more specific roles."""
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    # Check for more specific role matches first
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    # If no specific match, return Specialist
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.
    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major

# NEW: Director identification function
def identify_director_role(text: str, proposed_major: str, job_title: str = '') -> str:
    """Identify Director roles that may have been misclassified as Manager."""
    if proposed_major == 'Director':
        return proposed_major
    
    # Only check Manager roles for potential Director misclassification
    if proposed_major != 'Manager':
        return proposed_major
    
    t = (text or '').lower()
    title_lower = (job_title or '').lower()
    
    # Check for Director indicators with weighted scoring
    director_score = 0
    
    # Title contains Director/Dir (highest weight)
    if re.search(r'\b(director|dir)\b', title_lower):
        director_score += 3
    
    # District-wide scope
    if re.search(r'district[- ]?wide|all\s+schools|entire\s+district', t):
        director_score += 2
    
    # Large workforce (100+ employees)
    if re.search(r'([1-9]\d{2,}|\d{3,})\+?\s*(employees|staff|personnel|workforce)', t):
        director_score += 2
    
    # Reports to executive leadership
    if re.search(r'reports?\s+to\s+(chief|deputy|superintendent|board|executive)', t):
        director_score += 2
    
    # Budget authority ($1M+)
    if re.search(r'(million|MM|\$[1-9]\d{6,}|significant\s+budget|capital\s+projects?)', t):
        director_score += 1
    
    # Policy creation
    if re.search(r'(develop|create|establish|set)\s+(policy|policies|strategic\s+vision|direction)', t):
        director_score += 1
    
    # If score >= 3, it's likely a Director
    if director_score >= 3:
        return 'Director'
    
    # Special case: 200+ employees is strong Director signal
    large_workforce_match = re.search(r'([2-9]\d{2,}|\d{4,})\+?\s*(employees|staff)', t)
    if large_workforce_match:
        workforce_size = int(re.search(r'\d+', large_workforce_match.group()).group())
        if workforce_size >= 200:
            return 'Director'
    
    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Director and Principal typically get III for senior roles
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    return minor_role

print("✅ Enhanced closed sets and normalization helpers with Director fix defined")

In [ ]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)
    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")

    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")

In [ ]:
# ==== 5) Enhanced Zero Shot Prompt with Director Fix ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.
Process:
- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.
- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.
- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.
- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.
- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.
- Never justify classifications based on job titles - only use job attributes and MNPS standards.
IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet):
ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain, but prefer more specific roles when possible
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting
- **Technician vs Skilled Laborer**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Skilled Laborer: Physical, trade-based work to build, move, repair, or maintain the physical environment using hand or power tools
- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management
- **Supervisor vs Manager**: 
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree
- **Director vs Manager vs Supervisor**:
  * Director: District-wide or multi-department oversight, 100+ employees, reports to Executive 
    (Executive, Deputy, Chief), creates policy & strategic vision, $1M+ budget authority,
    represents to board/external stakeholders
  * Manager: Department or program-level oversight, 10-50 employees, reports to Directors,
    implements strategy & policy, <$1M operational budget, internal focus
  * Supervisor: Team-level oversight, frontline staff supervision, reports to Managers,
    ensures compliance with procedures, limited budget authority, may not require degree
- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming
MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity
Output Requirements:
- Provide only the new grouping for each job based on job function analysis.
- Ensure the grouping reflects the true nature of the work performed.
- Apply Problem Role Cheat Sheet guidelines to resolve ambiguous cases.
- Do NOT include or reference the original job title in your grouping decision or justification.
"""

print("✅ Enhanced zero shot prompt with Director distinction created")

In [ ]:
# ==== 6) LLM Setup and Helper Functions ====
from tqdm import tqdm
from google.colab import userdata

# OpenAI API setup - using Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ OpenAI API key loaded from Colab secrets")

# Model configuration
MODEL_ID = "gpt-4o-2024-11-20"

def call_llm_json_with_retry(prompt: str, model_id: str, max_retries: int = 3) -> dict:
    """Call LLM with JSON mode and retry logic."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "You are an expert HR analyst specializing in job classification."},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.2,
                max_tokens=1000
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            time.sleep(2 ** attempt)  # Exponential backoff

print("✅ OpenAI client configured with rate limiting protection")

# Self-consistency prompt for Pass 2
self_consistency_prompt = """
Review your previous classification and ensure consistency between your justification and selected role.

Check for these common issues:
1. If justification mentions "Manager" characteristics but classification is "Coordinator" 
2. If justification mentions "Coordinator" work but classification is "Manager"
3. If justification mentions "Coach" activities but classification is something else
4. If justification mentions "Director" level responsibilities but classification is "Manager"
5. Level mismatch: justification describes senior work but level is "I"

If there's a mismatch, correct the classification to match your justification.
If both are correct, return them unchanged.
"""

print("✅ Self-consistency prompt for two-pass approach defined")

In [ ]:
# ==== 7) Two-Pass Processing with Director Fix ====
def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description through two-pass classification with Director fix."""
    
    # Build job text from all available fields
    job_fields = ['Position Summary', 'Essential Functions', 'Knowledge', 
                  'Skills', 'Abilities', 'Education', 'Work Experience', 
                  'Licenses/Certifications']
    
    job_text = ""
    for field in job_fields:
        if field in row and pd.notna(row[field]):
            job_text += f"{field}: {row[field]}\n"
    
    # === PASS 1: Initial Classification ===
    pass1_prompt = f"""{zero_shot_prompt}

Available MNPS Major Roles (use ONLY these):
{', '.join(MNPS_MAJOR_ROLES)}

Job Description Attributes:
{job_text}

MNPS KSACs for Reference:
{KSACS_TEXT[:3000]}...

Critical reminders:
- Ignore the original job title completely
- Use ONLY approved MNPS major roles listed above
- Minor sub-group must be: I, II, III, or Lead
- Apply Director vs Manager distinction for executive roles
- Use Architect (Facility-Focused) for building/construction roles
- Use Architect (Technology-Focused) for system/software roles
- Executive roles (Coordinator, Principal, Director, Manager) rarely have "Lead" minor sub-grouping
- Ensure new_job_title incorporates both major_role_group and minor_sub_group
- Provide detailed justification that aligns with your selected role and level
Return your response as a JSON object with the following structure:
{{
  "new_job_title": "Descriptive title incorporating major_role_group and minor_sub_group",
  "major_role_group": "One of the approved MNPS roles",
  "minor_sub_group": "I, II, III, or Lead (consider executive role guidelines)",
  "grouping_justification": "Detailed explanation based on job attributes and KSACs alignment that matches your selected role and level"
}}"""
    
    try:
        # Get initial prediction
        pass1_response = call_llm_json_with_retry(pass1_prompt, MODEL_ID)
        
        # Apply post-processing logic
        major_role = pass1_response.get('major_role_group', 'Other')
        minor_role = pass1_response.get('minor_sub_group', 'I')
        justification = pass1_response.get('grouping_justification', 'No justification provided')
        new_job_title = pass1_response.get('new_job_title', f"{major_role} {minor_role}")
        
        # Apply rule-based corrections
        major_role = discourage_specialist(job_text, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)
        
        # NEW: Apply Director identification fix
        major_role = identify_director_role(job_text, major_role, row.get('Job Title', ''))
        
        minor_role = normalize_minor(minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        
        if not new_job_title or new_job_title == 'Unknown':
            new_job_title = f"{major_role} {minor_role}"

        # Reconstruct clean Pass 1 output
        pass1_clean = {
            "new_job_title": new_job_title,
            "major_role_group": major_role,
            "minor_sub_group": minor_role,
            "grouping_justification": justification
        }

        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = f"""{self_consistency_prompt}

**Your Previous Output:**
{json.dumps(pass1_clean, indent=2)}

**Now perform the self-consistency check and return the corrected (or unchanged) JSON.**
"""
        # Call LLM again for consistency check
        pass2_response = call_llm_json_with_retry(pass2_prompt, MODEL_ID)

        # Extract final values (do NOT re-apply post-processing to avoid overriding LLM correction)
        final_major = pass2_response.get('major_role_group', major_role)
        final_minor = pass2_response.get('minor_sub_group', minor_role)
        final_title = pass2_response.get('new_job_title', new_job_title)
        final_justification = pass2_response.get('grouping_justification', justification)

        # Re-normalize minor role (in case LLM outputs "1", etc.)
        final_minor = normalize_minor(final_minor)

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': final_title,
            'major_role_group': final_major,
            'minor_sub_group': final_minor,
            'grouping_justification': final_justification,
            'model_used': MODEL_ID
        }

    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions with rate limiting protection
results = []
print("🚀 Starting two-pass batch processing with Director fix and rate limiting...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    # Add small delay between requests to prevent rate limiting
    time.sleep(0.2)

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v755_director_fix.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

In [ ]:
# ==== 8) Generate Summary Statistics and Examples ====
# Load the results
preds = results_df.copy()

# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

# Create summary
summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 
               'director_count', 'manager_count', 'specialist_count', 
               'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Director').sum()),
        int((preds['major_role_group'] == 'Manager').sum()),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})
summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v755_director_fix.csv"
summary_stats.to_csv(summary_path, index=False)

# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title', 
                  'major_role_group', 'minor_sub_group']].head(10)
examples_path = OUTPUTS_DIR / "examples_gpt4o_v755_director_fix.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))
print("\n📝 Major Role Distribution:")
print(major_counts.to_string())
print("\n📝 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n🎯 Director Classifications:")
director_jobs = preds[preds['major_role_group'] == 'Director']
if len(director_jobs) > 0:
    print(f"Found {len(director_jobs)} Director classifications:")
    print(director_jobs[['job_title_original', 'new_job_title', 'minor_sub_group']].to_string(index=False))
else:
    print("No Director classifications found")
print("\n📝 Example Classifications:")
print(examples.to_string(index=False))
print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")

In [ ]:
# ==== 9) Enhanced Quality Check and Validation ====
# Check for alignment issues between justification and selected roles
def check_justification_alignment(row):
    """Check if the justification mentions the classified role."""
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    
    # Check if the major role is mentioned in the justification
    if major_role in justification:
        return True
    
    # Check for common variations
    role_variations = {
        'coordinator': ['coordination', 'coordinate'],
        'manager': ['management', 'managing'],
        'director': ['directing', 'direction'],
        'analyst': ['analysis', 'analytical'],
        'specialist': ['specialization', 'specialized'],
        'coach': ['coaching', 'mentoring']
    }
    
    if major_role in role_variations:
        for variant in role_variations[major_role]:
            if variant in justification:
                return True
    
    return False

# Apply alignment check
preds['justification_aligned'] = preds.apply(check_justification_alignment, axis=1)
alignment_rate = preds['justification_aligned'].mean() * 100

print(f"\n📊 Quality Metrics:")
print(f"Justification alignment rate: {alignment_rate:.1f}%")

# Show misaligned cases
misaligned = preds[~preds['justification_aligned']]
if len(misaligned) > 0:
    print(f"\n⚠️  Found {len(misaligned)} cases with potential misalignment:")
    print(misaligned[['source_row_index', 'job_title_original', 'major_role_group']].head())

# Save quality report
quality_report = pd.DataFrame({
    'metric': ['total_classifications', 'alignment_rate', 'misaligned_count', 
               'director_classifications', 'manager_classifications'],
    'value': [len(preds), f"{alignment_rate:.1f}%", len(misaligned),
              (preds['major_role_group'] == 'Director').sum(),
              (preds['major_role_group'] == 'Manager').sum()]
})
quality_path = OUTPUTS_DIR / "quality_report_gpt4o_v755_director_fix.csv"
quality_report.to_csv(quality_path, index=False)
print(f"\n✅ Saved quality report to: {quality_path}")

In [ ]:
# ==== 10) Comparison with Ground Truth (if available) ====
if 'major_role_group' in gt_df.columns:
    # Merge predictions with ground truth
    comparison = preds.merge(
        gt_df[['source_row_index', 'major_role_group', 'minor_sub_group']],
        on='source_row_index',
        suffixes=('_pred', '_gt'),
        how='inner'
    )
    
    # Calculate accuracy
    if len(comparison) > 0:
        major_accuracy = (comparison['major_role_group_pred'] == comparison['major_role_group_gt']).mean() * 100
        minor_accuracy = (comparison['minor_sub_group_pred'] == comparison['minor_sub_group_gt']).mean() * 100
        combined_accuracy = ((comparison['major_role_group_pred'] == comparison['major_role_group_gt']) & 
                           (comparison['minor_sub_group_pred'] == comparison['minor_sub_group_gt'])).mean() * 100
        
        print(f"\n🎯 Accuracy vs Ground Truth:")
        print(f"Major Role Group: {major_accuracy:.1f}%")
        print(f"Minor Sub Group: {minor_accuracy:.1f}%")
        print(f"Combined (Both): {combined_accuracy:.1f}%")
        
        # Check Director classifications specifically
        director_gt = comparison[comparison['major_role_group_gt'] == 'Director']
        if len(director_gt) > 0:
            director_accuracy = (director_gt['major_role_group_pred'] == 'Director').mean() * 100
            print(f"\n🎯 Director Classification Accuracy: {director_accuracy:.1f}%")
            print(f"Director cases in ground truth: {len(director_gt)}")
            
            # Show Director misclassifications
            director_wrong = director_gt[director_gt['major_role_group_pred'] != 'Director']
            if len(director_wrong) > 0:
                print(f"\n❌ Director Misclassifications:")
                print(director_wrong[['job_title_original', 'major_role_group_pred']].to_string(index=False))
        
        # Save comparison
        comparison_path = OUTPUTS_DIR / "comparison_vs_ground_truth_v755.csv"
        comparison.to_csv(comparison_path, index=False)
        print(f"\n✅ Saved comparison to: {comparison_path}")
else:
    print("\n⚠️  Ground truth classifications not available for comparison")

print("\n" + "="*60)
print("🎉 MNPS Job Classification v7.5.5 (Director Fix) Complete!")
print(f"📁 All outputs saved to: {OUTPUTS_DIR}")
print("="*60)